# Stage 04 — Exploratory Rheology Analysis

**Project:** Curation, Quality Control and Reproducible Analysis of Collagen-Hydrogel Rheology Data  
**Material:** bovine-collagen hydrogels  
**Source dataset:** Zenodo, DOI [10.5281/zenodo.17413651](https://doi.org/10.5281/zenodo.17413651)

This notebook explores the validated Stage 03 measurement tables. It does not repeat the source-workbook extraction. Its purpose is to describe the observed frequency- and time-sweep patterns while keeping experiment-level replication, provenance, missingness and uncertainty visible.

## Scientific guardrails

- Measurement points are repeated observations within an experiment, not independent hydrogel samples.
- `measurement_point` is the source-reported measurement order. The supplied files do not establish elapsed time or the interval between points, so this notebook never converts it to seconds/minutes or estimates a rate.
- Frequency-sweep and time-sweep experiments remain separate. Matching concentration and replicate labels do not prove that the tests used the same physical specimen.
- The documented frequency-sweep observation `G′ = 0 Pa` is retained. A derived `tan δ = G″/G′` is undefined there and is represented as missing.
- QC decisions from Stage 03 are treated as provenance. No source measurements are silently changed, imputed, smoothed or removed.
- Concentration comparisons are descriptive. No hypothesis tests are used, and patterns are not interpreted as causal effects.

## Inputs and outputs

The notebook expects these Stage 03 files inside the project:

- `data/processed/frequency_sweep_standardised.csv`
- `data/processed/time_sweep_standardised.csv`
- `metadata/experiment_register.csv`
- `metadata/measurement_data_dictionary.csv`
- `data/quality_control/data_quality_issue_log.csv`

The two processed measurement CSVs should remain local unless redistribution rights are confirmed. This notebook contains workflow code and does not embed or redistribute those row-level data.

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

## 2. Find the project files and load the validated inputs

The path search checks the current Jupyter folder and its parent folders for the expected project layout. It raises a clear error if the notebook is not being run from within the project tree.

In [ ]:
required_paths = [
    Path("data/processed/frequency_sweep_standardised.csv"),
    Path("data/processed/time_sweep_standardised.csv"),
    Path("metadata/experiment_register.csv"),
    Path("metadata/measurement_data_dictionary.csv"),
    Path("data/quality_control/data_quality_issue_log.csv"),
]
project_root = next((root for root in [Path.cwd(), *Path.cwd().parents]
                     if all((root / rel).is_file() for rel in required_paths)), None)
if project_root is None:
    raise FileNotFoundError(
        f"Could not find the five Stage 03 inputs from {Path.cwd()}. "
        "Run this notebook from the project folder or its notebooks/ subfolder."
    )
input_paths = {path.name: project_root / path for path in required_paths}
print(f"Project root: {project_root}")
for name, path in input_paths.items():
    print(f"{name}: {path.relative_to(project_root)}")

frequency = pd.read_csv(input_paths["frequency_sweep_standardised.csv"])
time = pd.read_csv(input_paths["time_sweep_standardised.csv"])
experiments = pd.read_csv(input_paths["experiment_register.csv"])
data_dictionary = pd.read_csv(input_paths["measurement_data_dictionary.csv"])
qc_log = pd.read_csv(input_paths["data_quality_issue_log.csv"])
for name, frame in {"frequency": frequency, "time": time, "experiments": experiments,
                    "data dictionary": data_dictionary, "QC log": qc_log}.items():
    print(f"{name}: {frame.shape[0]} rows × {frame.shape[1]} columns")

## 3. Confirm schemas, expected coverage and experiment design

The field list comes from the attached data dictionary. These assertions check the notebook inputs against the completed Stage 03 validation, rather than re-performing extraction.

In [ ]:
measurement_columns = data_dictionary["field_name"].tolist()
assert len(measurement_columns) == 11
assert frequency.columns.tolist() == measurement_columns
assert time.columns.tolist() == measurement_columns
assert len(frequency) == 210 and frequency["experiment_id"].nunique() == 10
assert len(time) == 450 and time["experiment_id"].nunique() == 10
assert experiments["experiment_id"].nunique() == 20
assert qc_log["issue_id"].is_unique
print("Measurement schemas, row counts and experiment counts match the supplied Stage 03 files.")
display(data_dictionary[["field_name", "standardised_unit", "expected_in_test_types", "missing_value_policy"]])
print("Experiments by test type and concentration:")
display(experiments.groupby(["test_type_standard", "collagen_concentration_mg_ml"])
        .agg(experiments=("experiment_id", "nunique"),
             replicate_ids=("replicate_id", lambda s: ", ".join(map(str, sorted(s.unique())))))
        .reset_index())

### Attach registered experiment metadata

A many-to-one merge adds concentration, replicate label and test type from the experiment register. Replicate labels describe registered experiments; they do not establish physical sample identity.

In [ ]:
experiment_metadata = experiments[["experiment_id", "collagen_concentration_mg_ml", "replicate_id", "test_type_standard"]]
frequency = frequency.merge(experiment_metadata, on="experiment_id", how="left", validate="many_to_one", indicator="_register_match")
time = time.merge(experiment_metadata, on="experiment_id", how="left", validate="many_to_one", indicator="_register_match")
assert (frequency["_register_match"] == "both").all()
assert (time["_register_match"] == "both").all()
assert (frequency["test_type_standard"] == "frequency_sweep").all()
assert (time["test_type_standard"] == "time_sweep").all()
frequency = frequency.sort_values(["collagen_concentration_mg_ml", "replicate_id", "measurement_point"]).reset_index(drop=True)
time = time.sort_values(["collagen_concentration_mg_ml", "replicate_id", "measurement_point"]).reset_index(drop=True)
print("All 660 measurements match one registered experiment and the expected test type.")

## 4. Integrity, coverage and missingness

In [ ]:
for name, frame in [("Frequency", frequency), ("Time", time)]:
    print(f"\n{name} table: {len(frame)} rows; {frame['experiment_id'].nunique()} experiments")
    print("Unique measurement IDs:", frame["measurement_id"].nunique())
    print("Duplicate measurement IDs:", frame["measurement_id"].duplicated().sum())
    print("Duplicate experiment/source-row pairs:", frame.duplicated(["experiment_id", "source_excel_row"]).sum())
    print("Duplicate experiment/point pairs:", frame.duplicated(["experiment_id", "measurement_point"]).sum())
    display(frame[measurement_columns].isna().sum().rename("missing_count").to_frame())

print("Rows per experiment:")
rows_by_experiment = pd.concat([
    frequency.groupby("experiment_id").size().rename("rows").to_frame().assign(test_type="frequency_sweep"),
    time.groupby("experiment_id").size().rename("rows").to_frame().assign(test_type="time_sweep"),
]).reset_index()
display(rows_by_experiment.groupby(["test_type", "rows"]).size().rename("experiment_count").to_frame())

Frequency-sweep `shear_stress_pa`, `strain_percent` and `complex_viscosity_pa_s` are structurally absent for all 210 frequency rows, as specified in the data dictionary. These are not zeros or failed measurements. All 450 time-sweep rows have values in those fields.

## 5. Review Stage 03 quality-control decisions

In [ ]:
qc_columns = ["issue_id", "original_filename", "cell_or_range", "variable", "issue_category",
               "scientific_interpretation", "processing_decision", "resolution_status"]
display(qc_log[qc_columns])
print("QC issue counts:")
display(qc_log["issue_category"].value_counts().rename_axis("issue_category").to_frame("issue_count"))

zero_frequency = frequency.loc[frequency["storage_modulus_pa"].eq(0)].copy()
assert len(zero_frequency) == 1, f"Expected one documented G′=0 row; found {len(zero_frequency)}."
zero_row = zero_frequency.iloc[0]
print("\nRetained zero G′ observation:")
print({key: zero_row[key] for key in ["measurement_id", "experiment_id", "source_excel_row",
                                     "measurement_point", "angular_frequency_rad_s",
                                     "storage_modulus_pa", "loss_modulus_pa", "temperature_c"]})
assert {"VALUE-001", "SCHEMA-001"}.issubset(set(qc_log["issue_id"]))
print("QC checks passed. The undocumented formula column is not part of the canonical schema.")

## 6. Frequency-sweep response

Each of the 10 frequency experiments contains 21 ordered frequency points. The experiments share the same recorded grid (0.628–62.8 rad/s). Thin lines show individual experiments; bold lines show medians across experiments at each concentration and frequency. The bold curves are descriptive summaries, not inferential estimates. The x-axis is logarithmic; modulus axes remain linear so the documented zero `G′` is visible.

In [ ]:
frequency_grid = frequency.groupby("experiment_id")["angular_frequency_rad_s"].apply(tuple)
assert frequency_grid.nunique() == 1, "Frequency grids differ; review before aggregating."
concentrations = sorted(frequency["collagen_concentration_mg_ml"].unique())
colour_map = {0.8: "#2C7FB8", 1.5: "#F28E2B", 2.3: "#59A14F"}
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
for ax, column, ylabel, title in [
    (axes[0], "storage_modulus_pa", "Storage modulus, G′ (Pa)", "Storage modulus"),
    (axes[1], "loss_modulus_pa", "Loss modulus, G″ (Pa)", "Loss modulus"),
]:
    for concentration in concentrations:
        group = frequency.loc[frequency["collagen_concentration_mg_ml"].eq(concentration)]
        colour = colour_map[concentration]
        for _, exp in group.groupby("experiment_id"):
            exp = exp.sort_values("angular_frequency_rad_s")
            ax.plot(exp["angular_frequency_rad_s"], exp[column], color=colour, alpha=.30, lw=1)
        median = group.groupby("angular_frequency_rad_s")[column].median().sort_index()
        ax.plot(median.index, median.values, color=colour, lw=2.7,
                label=f"{concentration:g} mg/mL (n={group['experiment_id'].nunique()} experiments)")
    ax.set_xscale("log")
    ax.set_xlabel("Angular frequency, ω (rad/s)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)
fig.suptitle("Frequency-sweep response by collagen concentration", y=1.02, fontsize=14)
fig.tight_layout()
plt.show()

Within-concentration trace spread describes variation between registered experiments. Concentration-level curves are observational summaries: the available data do not establish that concentration alone caused any difference. `G′ = 0 Pa` remains visible at the lowest frequency.

### Descriptive range at every concentration and frequency

In [ ]:
frequency_summary = (frequency.groupby(["collagen_concentration_mg_ml", "angular_frequency_rad_s"])
    .agg(experiments=("experiment_id", "nunique"),
         G_prime_median_pa=("storage_modulus_pa", "median"), G_prime_min_pa=("storage_modulus_pa", "min"),
         G_prime_max_pa=("storage_modulus_pa", "max"), G_double_prime_median_pa=("loss_modulus_pa", "median"),
         G_double_prime_min_pa=("loss_modulus_pa", "min"), G_double_prime_max_pa=("loss_modulus_pa", "max"))
    .reset_index())
display(frequency_summary.head(9))
print("One summary row represents a concentration × measured-frequency combination; n counts experiments.")

## 7. Derived loss factor, tan δ

The loss factor is calculated as `tan δ = G″ / G′`. Both source fields are in pascals, so the ratio is dimensionless. It is a derived quantity, not a source-reported field. When `G′ = 0`, division is undefined; the derived value is set to `NaN`, while source measurements remain unchanged.

In [ ]:
frequency["tan_delta"] = np.divide(
    frequency["loss_modulus_pa"], frequency["storage_modulus_pa"],
    out=np.full(len(frequency), np.nan, dtype=float),
    where=frequency["storage_modulus_pa"].to_numpy() != 0,
)
assert frequency["tan_delta"].isna().sum() == 1
assert frequency.loc[frequency["storage_modulus_pa"].eq(0), "tan_delta"].isna().all()
fig, ax = plt.subplots(figsize=(8, 5))
for concentration in concentrations:
    group = frequency.loc[frequency["collagen_concentration_mg_ml"].eq(concentration)]
    colour = colour_map[concentration]
    for _, exp in group.groupby("experiment_id"):
        exp = exp.sort_values("angular_frequency_rad_s")
        ax.plot(exp["angular_frequency_rad_s"], exp["tan_delta"], color=colour, alpha=.28, lw=1)
    median = group.groupby("angular_frequency_rad_s")["tan_delta"].median().sort_index()
    ax.plot(median.index, median.values, color=colour, lw=2.7,
            label=f"{concentration:g} mg/mL (n={group['experiment_id'].nunique()})")
ax.set_xscale("log")
ax.set_xlabel("Angular frequency, ω (rad/s)")
ax.set_ylabel("Loss factor, tan δ (dimensionless)")
ax.set_title("Derived loss factor across the frequency sweep")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
print("Undefined tan δ values caused by G′=0:", frequency["tan_delta"].isna().sum())

At a measured point, `tan δ > 1` means the reported `G″` exceeds `G′`; `tan δ < 1` means the reported `G′` exceeds `G″`. The ratio alone does not identify a full material structure or prove a gelation mechanism.

## 8. Compare experiments at a shared frequency

All frequency experiments contain 6.28 rad/s. The following table compares experiment-level observations at this single shared measured frequency and shows the number of experiments and observed range. No hypothesis test is applied.

In [ ]:
reference_frequency = 6.28
at_reference = frequency.loc[frequency["angular_frequency_rad_s"].eq(reference_frequency)]
assert at_reference["experiment_id"].nunique() == 10
reference_summary = (at_reference.groupby("collagen_concentration_mg_ml")
    .agg(experiments=("experiment_id", "nunique"),
         G_prime_median_pa=("storage_modulus_pa", "median"), G_prime_min_pa=("storage_modulus_pa", "min"),
         G_prime_max_pa=("storage_modulus_pa", "max"), G_double_prime_median_pa=("loss_modulus_pa", "median"),
         G_double_prime_min_pa=("loss_modulus_pa", "min"), G_double_prime_max_pa=("loss_modulus_pa", "max"),
         tan_delta_median=("tan_delta", "median"))
    .reset_index())
display(reference_summary)
print(f"Shared measured frequency: {reference_frequency:g} rad/s.")

## 9. Time-sweep sequence by source measurement order

The data dictionary defines `measurement_point` as source-reported measurement order, not elapsed time. Therefore the x-axis is **measurement point (source order)**. These plots describe the recorded sequence only; they do not estimate duration, kinetic rates or a time-dependent mechanism.

Thin lines are individual experiments. The bold line is the median across experiments at the same concentration and point. Shading gives the observed minimum-to-maximum across experiments, not a confidence interval.

In [ ]:
time_responses = [
    ("storage_modulus_pa", "Storage modulus, G′ (Pa)"),
    ("loss_modulus_pa", "Loss modulus, G″ (Pa)"),
    ("complex_viscosity_pa_s", "Source-reported complex viscosity (Pa·s)"),
    ("shear_stress_pa", "Source-reported shear stress (Pa)"),
]
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for ax, (column, ylabel) in zip(axes.flat, time_responses):
    for concentration in concentrations:
        group = time.loc[time["collagen_concentration_mg_ml"].eq(concentration)]
        colour = colour_map[concentration]
        for _, exp in group.groupby("experiment_id"):
            exp = exp.sort_values("measurement_point")
            ax.plot(exp["measurement_point"], exp[column], color=colour, alpha=.25, lw=.9)
        summary = group.groupby("measurement_point")[column].agg(["median", "min", "max"]).sort_index()
        ax.plot(summary.index, summary["median"], color=colour, lw=2.5,
                label=f"{concentration:g} mg/mL (n={group['experiment_id'].nunique()})")
        ax.fill_between(summary.index, summary["min"], summary["max"], color=colour, alpha=.10, linewidth=0)
    ax.set_xlabel("Measurement point (source order; not elapsed time)")
    ax.set_ylabel(ylabel)
    ax.legend(frameon=False, fontsize=8)
fig.suptitle("Time-sweep values across source-reported measurement order", y=1.01, fontsize=14)
fig.tight_layout()
plt.show()

### Recorded conditions and ranges

The following table checks recorded temperature, strain and angular-frequency coverage by concentration. Values are reported as supplied; their presence does not establish elapsed time.

In [ ]:
time_context = time.groupby("collagen_concentration_mg_ml").agg(
    experiments=("experiment_id", "nunique"), temperature_min_c=("temperature_c", "min"),
    temperature_max_c=("temperature_c", "max"), strain_min_percent=("strain_percent", "min"),
    strain_max_percent=("strain_percent", "max"), angular_frequency_min_rad_s=("angular_frequency_rad_s", "min"),
    angular_frequency_max_rad_s=("angular_frequency_rad_s", "max"),
).reset_index()
display(time_context)
assert time["angular_frequency_rad_s"].nunique() == 1
print("The recorded time-sweep angular frequency is constant; measurement order is not elapsed time.")

## 10. Experiment-level differences between first and last recorded points

To avoid treating 45 repeated points in one experiment as 45 replicates, this section first calculates point-1 and point-45 differences within each experiment. Only then are medians and ranges summarised across experiments. These are sequence-endpoint differences, not rates or time-normalised changes.

In [ ]:
time_endpoints = time.sort_values("measurement_point").groupby(
    ["experiment_id", "collagen_concentration_mg_ml", "replicate_id"], as_index=False
).agg(
    first_point=("measurement_point", "first"), last_point=("measurement_point", "last"),
    G_prime_first_pa=("storage_modulus_pa", "first"), G_prime_last_pa=("storage_modulus_pa", "last"),
    G_double_prime_first_pa=("loss_modulus_pa", "first"), G_double_prime_last_pa=("loss_modulus_pa", "last"),
    viscosity_first_pa_s=("complex_viscosity_pa_s", "first"), viscosity_last_pa_s=("complex_viscosity_pa_s", "last"),
)
assert (time_endpoints["first_point"] == 1).all() and (time_endpoints["last_point"] == 45).all()
time_endpoints["G_prime_change_pa"] = time_endpoints["G_prime_last_pa"] - time_endpoints["G_prime_first_pa"]
time_endpoints["G_double_prime_change_pa"] = time_endpoints["G_double_prime_last_pa"] - time_endpoints["G_double_prime_first_pa"]
time_endpoints["viscosity_change_pa_s"] = time_endpoints["viscosity_last_pa_s"] - time_endpoints["viscosity_first_pa_s"]
display(time_endpoints.sort_values(["collagen_concentration_mg_ml", "replicate_id"]))
endpoint_summary = time_endpoints.groupby("collagen_concentration_mg_ml").agg(
    experiments=("experiment_id", "nunique"), G_prime_change_median_pa=("G_prime_change_pa", "median"),
    G_prime_change_min_pa=("G_prime_change_pa", "min"), G_prime_change_max_pa=("G_prime_change_pa", "max"),
    G_double_prime_change_median_pa=("G_double_prime_change_pa", "median"),
    viscosity_change_median_pa_s=("viscosity_change_pa_s", "median"),
).reset_index()
display(endpoint_summary)

## 11. Interpretation and limitations

### What these data support

- The frequency-sweep data support descriptive comparison of `G′`, `G″` and derived `tan δ` over a shared measured angular-frequency grid, with experiments as the replicate units.
- The time-sweep data support descriptive comparison across the source-recorded measurement order at constant recorded angular frequency (6.28 rad/s). They do not support elapsed-time kinetics or rates.
- Each test type has four registered experiments at 0.8 mg/mL and three at 1.5 and 2.3 mg/mL. These small, unequal counts warrant descriptive language and careful visibility of experiment-level spread.
- Differences among concentration groups can be observed but cannot be attributed to concentration alone from this dataset. Other specimen, preparation, batch or instrument factors may contribute.
- Frequency and time experiments with matching replicate labels are not paired as measurements on the same physical specimen; physical sample linkage remains unresolved.
- Measurement rows are repeated observations within experiments. Treating all 660 rows as independent hydrogel samples would be pseudoreplication.

### What this notebook does not conclude

It does not interpolate unmeasured frequencies, fit a constitutive model, infer gelation kinetics, calculate time-dependent rates, pair different test types by replicate label or perform inferential significance tests. It does not impute the structural missingness in frequency-sweep fields.

In [ ]:
print("Frequency-sweep experiment summaries at 6.28 rad/s:")
display(reference_summary[["collagen_concentration_mg_ml", "experiments", "G_prime_median_pa",
                           "G_double_prime_median_pa", "tan_delta_median"]])
print("Time-sweep differences between measurement points 1 and 45:")
display(endpoint_summary)
print("Replicate counts are experiments, not measurement rows:")
display(experiments.groupby(["test_type_standard", "collagen_concentration_mg_ml"]).size().rename("experiment_count").to_frame())

## 12. Reproducibility and provenance

- Inputs are the five Stage 03 files listed above; paths are project-relative.
- Joins are explicitly validated as many-to-one, and all displayed grouping/sorting is deterministic.
- Source measurements are not overwritten. `tan_delta` is a separate in-memory derived field; undefined division remains missing.
- `measurement_point` remains source order and is never converted to time.
- The retained zero `G′` remains traceable through `measurement_id`, `experiment_id` and `source_excel_row`.
- The undocumented formula column noted in QC issue `SCHEMA-001` is not reintroduced.
- The notebook does not write or publish source workbooks or processed row-level measurements.

**Stage 04 status:** exploratory plots and descriptive summaries are generated from the validated Stage 03 files. Interpretation remains observational and subject to the limitations above.